# CPCV Energy Model Development

This notebook builds the modelling workflow step by step.

The aim is simple:

1. Use the energy tickers only: `cl1s`, `ho1s`, `rb1s`, `ng1s`.
2. Build triple-barrier labels for each ticker/config.
3. Join those labels to the feature matrix.
4. Use CPCV to test models and feature-selection methods.
5. Rank models by AUC.
6. Take the top 5 per ticker.
7. Choose the final model using the Sharpe distribution across CPCV paths.

The notebook starts with a small smoke run so the logic is easy to follow. Later, set `RUN_FULL_GRID = True` to run the larger search.

## 1. Imports

This cell imports the libraries and the triple-barrier function from `02_tripple_barrier/tripple_barrier.py`.

In [ ]:
import json
import os
import sys
import tempfile
import warnings
from itertools import combinations, product
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib"))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", exc)

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    shap = None
    SHAP_AVAILABLE = False
    print("SHAP unavailable, so SHAP feature selection will be skipped:", exc)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "02_tripple_barrier"))
from tripple_barrier import run_triple_barrier_pipeline

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## 2. Main Settings

Keep these settings small at first. The notebook will run only a small smoke test unless `RUN_FULL_GRID` is changed to `True`.

In [ ]:
ENERGY_TICKERS = ["cl1s", "ho1s", "rb1s", "ng1s"]
TRAINING_END = pd.Timestamp("2022-01-01")
RANDOM_STATE = 42

FEATURE_MATRIX_PATH = PROJECT_ROOT / "data" / "features" / "merged_feature_matrix.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "models" / "cpcv_energy"

# Start here. Keep False until the small run looks correct.
RUN_FULL_GRID = False
SAVE_OUTPUTS = False

# The smoke run is deliberately tiny and easy to inspect.
SMOKE_TICKERS = ["cl1s"]
PROBA_THRESHOLD = 0.50
TOP_K = 5
MIN_VALID_AUC_PATHS = 3

## 3. Load Features

We use the full merged feature matrix. No feature selection happens here. Selection happens inside each CPCV training fold.

In [ ]:
features = pd.read_csv(FEATURE_MATRIX_PATH, parse_dates=["date"])
features = features[features["instrument"].isin(ENERGY_TICKERS)].copy()
features = features[features["date"] < TRAINING_END].copy()
features = features.sort_values(["instrument", "date"]).reset_index(drop=True)

print("Feature matrix:", features.shape)
display(features.groupby("instrument").agg(rows=("date", "size"), min_date=("date", "min"), max_date=("date", "max")))
features.head()

## 4. Define Small And Full Grids

These grids control what gets tested. The smoke grid is small. The full grid is larger but still readable.

In [ ]:
smoke_barrier_grid = [
    {
        "name": "ewma_d10_tp2_sl2",
        "volatility_method": "ewma",
        "ewma_span": 100,
        "volatility_window": 20,
        "num_days": 10,
        "take_profit_mult": 2.0,
        "stop_loss_mult": 2.0,
    }
]

full_barrier_grid = []
for vol_method in ["ewma", "rolling", "parkinson", "garman_klass"]:
    for num_days in [5, 10, 20]:
        for tp, sl in [(1.5, 1.5), (2.0, 2.0), (3.0, 3.0), (2.0, 1.5), (1.5, 2.0)]:
            full_barrier_grid.append({
                "name": f"{vol_method}_d{num_days}_tp{tp}_sl{sl}",
                "volatility_method": vol_method,
                "ewma_span": 100,
                "volatility_window": 20,
                "num_days": num_days,
                "take_profit_mult": tp,
                "stop_loss_mult": sl,
            })

smoke_feature_selection_grid = [
    {"name": "none", "method": "none"},
    {"name": "mdi_top20", "method": "mdi", "k": 20},
    {"name": "cluster_corr90", "method": "cluster", "corr_threshold": 0.90, "max_features": 30},
    {"name": "pca90", "method": "pca", "variance": 0.90},
]
if SHAP_AVAILABLE:
    smoke_feature_selection_grid.append({"name": "shap_top20", "method": "shap", "k": 20})

full_feature_selection_grid = [
    {"name": "none", "method": "none"},
    {"name": "mdi_top20", "method": "mdi", "k": 20},
    {"name": "mdi_top50", "method": "mdi", "k": 50},
    {"name": "cluster_corr90", "method": "cluster", "corr_threshold": 0.90, "max_features": 50},
    {"name": "cluster_corr95", "method": "cluster", "corr_threshold": 0.95, "max_features": 75},
    {"name": "pca80", "method": "pca", "variance": 0.80},
    {"name": "pca90", "method": "pca", "variance": 0.90},
    {"name": "pca95", "method": "pca", "variance": 0.95},
    {"name": "cluster90_pca90", "method": "cluster_pca", "corr_threshold": 0.90, "max_features": 75, "variance": 0.90},
]
if SHAP_AVAILABLE:
    full_feature_selection_grid.extend([
        {"name": "shap_top20", "method": "shap", "k": 20},
        {"name": "shap_top50", "method": "shap", "k": 50},
    ])

smoke_model_grid = [
    {
        "name": "logistic",
        "needs_scaling": True,
        "model": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    },
    {
        "name": "random_forest",
        "needs_scaling": False,
        "model": RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1),
    },
]

full_model_grid = smoke_model_grid + [
    {
        "name": "extra_trees",
        "needs_scaling": False,
        "model": ExtraTreesClassifier(n_estimators=150, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    },
    {
        "name": "hist_gradient_boosting",
        "needs_scaling": False,
        "model": HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=15, random_state=RANDOM_STATE),
    },
    {
        "name": "mlp",
        "needs_scaling": True,
        "model": MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, early_stopping=True, random_state=RANDOM_STATE),
    },
]
if XGBOOST_AVAILABLE:
    full_model_grid.append({
        "name": "xgboost",
        "needs_scaling": False,
        "model": XGBClassifier(n_estimators=120, max_depth=2, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    })

if RUN_FULL_GRID:
    tickers_to_run = ENERGY_TICKERS
    barrier_grid = full_barrier_grid
    feature_selection_grid = full_feature_selection_grid
    model_grid = full_model_grid
else:
    tickers_to_run = SMOKE_TICKERS
    barrier_grid = smoke_barrier_grid
    feature_selection_grid = smoke_feature_selection_grid
    model_grid = smoke_model_grid

print("Tickers:", tickers_to_run)
print("Barrier configs:", len(barrier_grid))
print("Feature selectors:", len(feature_selection_grid))
print("Models:", len(model_grid))
print("Candidates per ticker:", len(barrier_grid) * len(feature_selection_grid) * len(model_grid))

## 5. Build Labels And Modelling Data

For a ticker and one triple-barrier config, we:

1. Generate labels.
2. Join labels to the feature matrix.
3. Keep only numeric feature columns.
4. Remove label/outcome columns so the model cannot cheat.

In [ ]:
label_cache = {}

leakage_columns = {
    "training_end", "vol", "tp", "sl", "timeout_date", "timeout_close",
    "touch_date", "touch_price", "touched_barrier", "triple_barrier_label",
    "metalabel", "holding_period_days", "raw_touch_return", "signed_touch_return",
    "volatility_method", "ewma_span", "volatility_window", "num_days",
    "take_profit_mult", "stop_loss_mult",
}


def get_labels(ticker, barrier_config):
    key = (ticker, tuple(sorted(barrier_config.items())))
    if key not in label_cache:
        kwargs = {k: v for k, v in barrier_config.items() if k != "name"}
        label_cache[key] = run_triple_barrier_pipeline(
            instrument=ticker,
            training_end=TRAINING_END,
            **kwargs,
        )
    return label_cache[key].copy()


def make_model_data(ticker, barrier_config):
    labels = get_labels(ticker, barrier_config)
    ticker_features = features[features["instrument"] == ticker].copy()

    # Use primary_signal from labels. Use close/OHLCV columns from the feature matrix.
    labels_for_join = labels.drop(columns=["close"], errors="ignore")
    ticker_features = ticker_features.drop(columns=["primary_signal"], errors="ignore")
    data = labels_for_join.merge(ticker_features, on=["date", "instrument"], how="inner")
    data = data.sort_values("date").reset_index(drop=True)

    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c not in leakage_columns]
    data[feature_cols] = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    return data, feature_cols


example_data, example_features = make_model_data(tickers_to_run[0], barrier_grid[0])
print("Rows:", len(example_data))
print("Feature count:", len(example_features))
print("Metalabel counts:")
print(example_data["metalabel"].value_counts().sort_index())
example_data.head()

## 6. CPCV Splits

This is a simple purged CPCV splitter.

It divides rows into chronological groups, picks test groups, removes overlapping training events, then applies an embargo.

In [ ]:
def make_groups(n_rows, n_groups):
    return [g.astype(int) for g in np.array_split(np.arange(n_rows), n_groups) if len(g) > 0]


def embargo_date(dates, end_date, embargo):
    unique_dates = pd.Series(pd.to_datetime(dates).sort_values().unique())
    pos = unique_dates.searchsorted(end_date, side="right")
    if pos >= len(unique_dates):
        return end_date
    return unique_dates.iloc[min(len(unique_dates) - 1, pos + embargo - 1)]


def make_cpcv_splits(data, n_groups=4, k_test_groups=1, embargo=5):
    data = data.sort_values("date").reset_index(drop=True)
    groups = make_groups(len(data), n_groups)
    splits = []

    for split_id, test_group_ids in enumerate(combinations(range(len(groups)), k_test_groups)):
        test_idx = np.concatenate([groups[i] for i in test_group_ids])
        train_mask = np.ones(len(data), dtype=bool)
        train_mask[test_idx] = False

        for group_id in test_group_ids:
            group_idx = groups[group_id]
            test_start = data.loc[group_idx, "date"].min()
            test_end = data.loc[group_idx, "date"].max()
            test_horizon_end = data.loc[group_idx, "timeout_date"].max()
            emb_end = embargo_date(data["date"], test_end, embargo)

            overlapping = (data["date"] <= test_horizon_end) & (data["timeout_date"] >= test_start)
            embargoed = (data["date"] > test_end) & (data["date"] <= emb_end)
            train_mask[overlapping | embargoed] = False

        train_idx = np.where(train_mask)[0]
        if len(train_idx) > 0 and len(test_idx) > 0:
            splits.append({
                "split_id": split_id,
                "test_groups": test_group_ids,
                "train_idx": train_idx,
                "test_idx": test_idx,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
            })
    return splits


# Smaller tickers have fewer active labels, so use fewer groups.
def cpcv_settings_for_ticker(ticker):
    if ticker in ["cl1s", "rb1s"]:
        return {"n_groups": 6, "k_test_groups": 2, "embargo": 5}
    return {"n_groups": 4, "k_test_groups": 1, "embargo": 5}


example_splits = make_cpcv_splits(example_data, n_groups=4, k_test_groups=1, embargo=5)
print("Number of example splits:", len(example_splits))
display(pd.DataFrame([{k: v for k, v in s.items() if k not in ["train_idx", "test_idx"]} for s in example_splits]))

## 7. Feature Selection Functions

These functions are intentionally plain. They receive already-imputed training and test data.

Important: every selector is fitted on the training fold only.

In [ ]:
def select_none(X_train, X_test, y_train, config):
    return X_train.values, X_test.values, list(X_train.columns), None


def select_mdi(X_train, X_test, y_train, config):
    k = min(config.get("k", 20), X_train.shape[1])
    tree = ExtraTreesClassifier(n_estimators=100, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
    tree.fit(X_train, y_train)
    importance = pd.Series(tree.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    selected = importance.head(k).index.tolist()
    report = importance.head(k).reset_index().rename(columns={"index": "feature", 0: "importance"})
    return X_train[selected].values, X_test[selected].values, selected, report


def select_shap(X_train, X_test, y_train, config):
    if not SHAP_AVAILABLE:
        raise RuntimeError("SHAP is not available in this environment.")

    k = min(config.get("k", 20), X_train.shape[1])
    tree = ExtraTreesClassifier(n_estimators=100, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
    tree.fit(X_train, y_train)

    sample = X_train.sample(min(200, len(X_train)), random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(tree)
    shap_values = explainer.shap_values(sample)

    if isinstance(shap_values, list):
        values = shap_values[1]
    elif getattr(shap_values, "ndim", 0) == 3:
        values = shap_values[:, :, 1]
    else:
        values = shap_values

    importance = pd.Series(np.abs(values).mean(axis=0), index=X_train.columns).sort_values(ascending=False)
    selected = importance.head(k).index.tolist()
    report = importance.head(k).reset_index().rename(columns={"index": "feature", 0: "importance"})
    return X_train[selected].values, X_test[selected].values, selected, report


def select_cluster(X_train, X_test, y_train, config):
    corr_threshold = config.get("corr_threshold", 0.90)
    max_features = config.get("max_features", 50)

    corr = X_train.corr().abs().fillna(0).clip(0, 1)
    distance = (1 - corr).clip(0, 1)
    np.fill_diagonal(distance.values, 0)

    linkage = hierarchy.linkage(squareform(distance.values, checks=False), method="average")
    clusters = hierarchy.fcluster(linkage, t=1 - corr_threshold, criterion="distance")

    mi = pd.Series(mutual_info_classif(X_train, y_train, random_state=RANDOM_STATE), index=X_train.columns)

    selected = []
    rows = []
    for cluster_id in sorted(np.unique(clusters)):
        members = [X_train.columns[i] for i, c in enumerate(clusters) if c == cluster_id]
        best = mi.loc[members].sort_values(ascending=False).index[0]
        selected.append(best)
        rows.append({"feature": best, "cluster": cluster_id, "score": mi.loc[best], "cluster_size": len(members)})

    report = pd.DataFrame(rows).sort_values("score", ascending=False)
    selected = report.head(max_features)["feature"].tolist()
    return X_train[selected].values, X_test[selected].values, selected, report.head(max_features)


def select_pca(X_train, X_test, y_train, config):
    variance = config.get("variance", 0.90)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    pca = PCA(n_components=variance, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    names = [f"PC{i+1}" for i in range(X_train_pca.shape[1])]
    report = pd.DataFrame({
        "component": names,
        "explained_variance": pca.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
    })
    return X_train_pca, X_test_pca, names, report


def select_cluster_pca(X_train, X_test, y_train, config):
    X_train_cluster, X_test_cluster, selected, cluster_report = select_cluster(X_train, X_test, y_train, config)
    X_train_cluster = pd.DataFrame(X_train_cluster, columns=selected, index=X_train.index)
    X_test_cluster = pd.DataFrame(X_test_cluster, columns=selected, index=X_test.index)
    X_train_pca, X_test_pca, names, pca_report = select_pca(X_train_cluster, X_test_cluster, y_train, config)
    return X_train_pca, X_test_pca, names, {"cluster_report": cluster_report, "pca_report": pca_report}


def apply_feature_selection(X_train, X_test, y_train, config):
    method = config["method"]
    if method == "none":
        return select_none(X_train, X_test, y_train, config)
    if method == "mdi":
        return select_mdi(X_train, X_test, y_train, config)
    if method == "shap":
        return select_shap(X_train, X_test, y_train, config)
    if method == "cluster":
        return select_cluster(X_train, X_test, y_train, config)
    if method == "pca":
        return select_pca(X_train, X_test, y_train, config)
    if method == "cluster_pca":
        return select_cluster_pca(X_train, X_test, y_train, config)
    raise ValueError(f"Unknown feature selection method: {method}")

## 8. Evaluate One Candidate

A candidate is one combination of:

- ticker
- triple-barrier config
- feature-selection config
- model config

The model is evaluated across CPCV splits.

In [ ]:
def get_prediction_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))


def sharpe_ratio(returns):
    returns = pd.Series(returns).dropna()
    if len(returns) < 2:
        return np.nan
    if returns.std(ddof=1) == 0:
        return np.nan
    return returns.mean() / returns.std(ddof=1) * np.sqrt(len(returns))


def evaluate_candidate(ticker, barrier_config, fs_config, model_config):
    data, feature_cols = make_model_data(ticker, barrier_config)
    settings = cpcv_settings_for_ticker(ticker)
    if not RUN_FULL_GRID:
        settings = {"n_groups": 4, "k_test_groups": 1, "embargo": 5}
    splits = make_cpcv_splits(data, **settings)

    rows = []
    first_feature_report = None

    for split in splits:
        train = data.iloc[split["train_idx"]]
        test = data.iloc[split["test_idx"]]

        y_train = train["metalabel"].astype(int)
        y_test = test["metalabel"].astype(int)

        if y_train.nunique() < 2:
            continue

        X_train_raw = train[feature_cols].replace([np.inf, -np.inf], np.nan)
        X_test_raw = test[feature_cols].replace([np.inf, -np.inf], np.nan)

        imputer = SimpleImputer(strategy="median")
        X_train = pd.DataFrame(imputer.fit_transform(X_train_raw), columns=feature_cols, index=train.index)
        X_test = pd.DataFrame(imputer.transform(X_test_raw), columns=feature_cols, index=test.index)

        try:
            X_train_sel, X_test_sel, selected_features, feature_report = apply_feature_selection(X_train, X_test, y_train, fs_config)
        except RuntimeError as exc:
            print(f"Skipping {fs_config['name']} for {ticker}: {exc}")
            return [], None

        if model_config["needs_scaling"]:
            scaler = StandardScaler()
            X_train_model = scaler.fit_transform(X_train_sel)
            X_test_model = scaler.transform(X_test_sel)
        else:
            X_train_model = X_train_sel
            X_test_model = X_test_sel

        model = clone(model_config["model"])
        model.fit(X_train_model, y_train)

        y_score = get_prediction_scores(model, X_test_model)
        y_pred = (y_score >= PROBA_THRESHOLD).astype(int)

        if y_test.nunique() == 2:
            auc = roc_auc_score(y_test, y_score)
        else:
            auc = np.nan

        trade_mask = y_pred == 1
        strategy_returns = test.loc[trade_mask, "signed_touch_return"]
        sharpe = sharpe_ratio(strategy_returns)

        rows.append({
            "ticker": ticker,
            "barrier": barrier_config["name"],
            "feature_selection": fs_config["name"],
            "model": model_config["name"],
            "split_id": split["split_id"],
            "auc": auc,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "sharpe": sharpe,
            "trade_count": int(trade_mask.sum()),
            "selected_feature_count": len(selected_features),
            "num_days": barrier_config["num_days"],
            "take_profit_mult": barrier_config["take_profit_mult"],
            "stop_loss_mult": barrier_config["stop_loss_mult"],
            "volatility_method": barrier_config["volatility_method"],
        })

        if first_feature_report is None:
            first_feature_report = {
                "ticker": ticker,
                "barrier": barrier_config["name"],
                "feature_selection": fs_config["name"],
                "model": model_config["name"],
                "selected_features": selected_features,
                "report": feature_report,
            }

    return rows, first_feature_report

## 9. Run The Grid

This loops through the chosen tickers and candidate grids.

With the default smoke settings, this should be quick.

In [ ]:
all_rows = []
feature_selection_reports = []

total_candidates = len(tickers_to_run) * len(barrier_grid) * len(feature_selection_grid) * len(model_grid)
print("Total candidates:", total_candidates)

candidate_number = 0
for ticker in tickers_to_run:
    for barrier_config, fs_config, model_config in product(barrier_grid, feature_selection_grid, model_grid):
        candidate_number += 1
        print(f"{candidate_number}/{total_candidates}: {ticker} | {barrier_config['name']} | {fs_config['name']} | {model_config['name']}")
        rows, report = evaluate_candidate(ticker, barrier_config, fs_config, model_config)
        all_rows.extend(rows)
        if report is not None:
            feature_selection_reports.append(report)

all_cpcv_results = pd.DataFrame(all_rows)
print("Path-level result shape:", all_cpcv_results.shape)
all_cpcv_results.head()

## 10. Summarise And Rank By AUC

We summarise each candidate across CPCV paths, then rank by mean AUC.

In [ ]:
def summarise_results(path_results):
    summary = (
        path_results
        .groupby(["ticker", "barrier", "feature_selection", "model"], as_index=False)
        .agg(
            mean_auc=("auc", "mean"),
            median_auc=("auc", "median"),
            valid_auc_paths=("auc", lambda x: x.notna().sum()),
            median_sharpe=("sharpe", "median"),
            mean_sharpe=("sharpe", "mean"),
            sharpe_iqr=("sharpe", lambda x: x.quantile(0.75) - x.quantile(0.25)),
            sharpe_std=("sharpe", "std"),
            total_trades=("trade_count", "sum"),
            avg_selected_features=("selected_feature_count", "mean"),
        )
    )
    summary = summary[summary["valid_auc_paths"] >= MIN_VALID_AUC_PATHS].copy()
    summary = summary.sort_values(["ticker", "mean_auc"], ascending=[True, False]).reset_index(drop=True)
    return summary

candidate_summary = summarise_results(all_cpcv_results)
print("Candidate summary shape:", candidate_summary.shape)
candidate_summary.head(20)

## 11. Top 5 By AUC And Sharpe Distribution

For each ticker, we take the top 5 by AUC. Then we inspect the Sharpe ratio distribution across CPCV paths.

In [ ]:
top5_by_ticker = {}
for ticker, group in candidate_summary.groupby("ticker"):
    top5_by_ticker[ticker] = group.head(TOP_K).copy()

for ticker, top5 in top5_by_ticker.items():
    print()
    print(f"Top {len(top5)} candidates for {ticker}")
    display(top5)

    plot_data = []
    labels = []
    for _, row in top5.iterrows():
        mask = (
            (all_cpcv_results["ticker"] == row["ticker"])
            & (all_cpcv_results["barrier"] == row["barrier"])
            & (all_cpcv_results["feature_selection"] == row["feature_selection"])
            & (all_cpcv_results["model"] == row["model"])
        )
        values = all_cpcv_results.loc[mask, "sharpe"].dropna().values
        plot_data.append(values)
        labels.append(chr(10).join([str(row["feature_selection"]), str(row["model"])]))

    if plot_data:
        plt.figure(figsize=(12, 5))
        plt.boxplot(plot_data, labels=labels, showmeans=True)
        plt.axhline(0, color="black", linewidth=1, alpha=0.5)
        plt.title(f"{ticker}: Sharpe distributions for top-{len(top5)} AUC configs")
        plt.ylabel("Sharpe across CPCV paths")
        plt.xticks(rotation=20, ha="right")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()

## 12. Pick Final Configs

Final selection rule:

1. Start from the top 5 AUC candidates.
2. Pick highest median Sharpe.
3. If close, prefer lower Sharpe IQR/spread.

In [ ]:
selected_configs = {}
for ticker, top5 in top5_by_ticker.items():
    ranked = top5.sort_values(
        ["median_sharpe", "sharpe_iqr", "sharpe_std", "mean_auc"],
        ascending=[False, True, True, False],
    ).reset_index(drop=True)
    selected_configs[ticker] = ranked.iloc[0].to_dict()

selected_configs_df = pd.DataFrame(selected_configs).T
selected_configs_df

## 13. Look At Feature Selection Reports

This cell shows what features were selected for the chosen configs when the selector is interpretable.

For PCA, the report is component explained variance rather than raw feature importance.

In [ ]:
def find_report_for_config(selected):
    for report in feature_selection_reports:
        if (
            report["ticker"] == selected["ticker"]
            and report["barrier"] == selected["barrier"]
            and report["feature_selection"] == selected["feature_selection"]
            and report["model"] == selected["model"]
        ):
            return report
    return None

for ticker, selected in selected_configs.items():
    print()
    print("Selected config for", ticker)
    report = find_report_for_config(selected)
    if report is None:
        print("No feature report found.")
        continue
    print("Feature selection:", report["feature_selection"])
    print("Selected feature count:", len(report["selected_features"]))
    if isinstance(report["report"], pd.DataFrame):
        display(report["report"].head(30))
    else:
        print(report["report"])

## 14. Fit Final Models

Now fit each selected configuration on all available pre-2022 labelled rows for that ticker.

These final fitted objects are kept in the `final_models` dictionary.

In [ ]:
def get_config_by_name(grid, name):
    for item in grid:
        if item["name"] == name:
            return item
    raise KeyError(name)


def fit_final_model(selected):
    ticker = selected["ticker"]
    barrier_config = get_config_by_name(barrier_grid, selected["barrier"])
    fs_config = get_config_by_name(feature_selection_grid, selected["feature_selection"])
    model_config = get_config_by_name(model_grid, selected["model"])

    data, feature_cols = make_model_data(ticker, barrier_config)
    y = data["metalabel"].astype(int)
    X_raw = data[feature_cols].replace([np.inf, -np.inf], np.nan)

    imputer = SimpleImputer(strategy="median")
    X = pd.DataFrame(imputer.fit_transform(X_raw), columns=feature_cols, index=data.index)

    X_selected, _, selected_features, fs_report = apply_feature_selection(X, X, y, fs_config)

    scaler = None
    if model_config["needs_scaling"]:
        scaler = StandardScaler()
        X_model = scaler.fit_transform(X_selected)
    else:
        X_model = X_selected

    model = clone(model_config["model"])
    model.fit(X_model, y)

    return {
        "ticker": ticker,
        "barrier_config": barrier_config,
        "feature_selection_config": fs_config,
        "model_config": {"name": model_config["name"], "needs_scaling": model_config["needs_scaling"]},
        "imputer": imputer,
        "scaler": scaler,
        "selected_features": selected_features,
        "feature_selection_report": fs_report,
        "model": model,
        "training_rows": len(data),
        "feature_cols_before_selection": feature_cols,
    }

final_models = {}
for ticker, selected in selected_configs.items():
    final_models[ticker] = fit_final_model(selected)

pd.DataFrame([
    {
        "ticker": ticker,
        "model": obj["model_config"]["name"],
        "feature_selection": obj["feature_selection_config"]["name"],
        "training_rows": obj["training_rows"],
        "selected_features": len(obj["selected_features"]),
    }
    for ticker, obj in final_models.items()
])

## 15. Optional Save

Nothing is saved unless `SAVE_OUTPUTS = True`.

In [ ]:
if SAVE_OUTPUTS:
    import joblib

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    all_cpcv_results.to_csv(OUTPUT_DIR / "path_level_results.csv", index=False)
    candidate_summary.to_csv(OUTPUT_DIR / "candidate_summary.csv", index=False)
    selected_configs_df.to_csv(OUTPUT_DIR / "selected_configs.csv")

    for ticker, obj in final_models.items():
        joblib.dump(obj, OUTPUT_DIR / f"{ticker}_final_model.joblib")

    print("Saved outputs to", OUTPUT_DIR)
else:
    print("SAVE_OUTPUTS is False, so nothing was written.")